# Porownanie modeli - Crash Severity Prediction

Notebook porownuje wyniki wszystkich modeli wytrenowanych w pipeline'ach Kedro:
- Random Forest (baseline)
- Gradient Boosting
- XGBoost
- LightGBM
- TPOT (AutoML)
- Optuna-tuned LightGBM

In [ ]:
import json
import pickle
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

sns.set_theme(style="whitegrid")

# Wczytanie wynikow z pipeline'ow
report_dir = Path("../data/08_reporting")
models_dir = Path("../data/06_models")

results = {}

# Baseline metrics
if (report_dir / "metrics.json").exists():
    with open(report_dir / "metrics.json") as f:
        m = json.load(f)
        results["Random Forest\n(baseline)"] = m

# Comparison metrics  
if (report_dir / "comparison_metrics.json").exists():
    with open(report_dir / "comparison_metrics.json") as f:
        comp = json.load(f)
        for name in ["RandomForest", "GradientBoosting", "XGBoost", "LightGBM"]:
            if name in comp:
                results[name] = comp[name]

# Tuning metrics
if (report_dir / "tuning_metrics.json").exists():
    with open(report_dir / "tuning_metrics.json") as f:
        m = json.load(f)
        results["Optuna-tuned\nLightGBM"] = {
            "accuracy": m.get("optuna_accuracy"),
            "f1_weighted": m.get("optuna_f1_weighted"),
            "f1_macro": m.get("optuna_f1_macro"),
        }

# AutoML metrics
if (report_dir / "automl_metrics.json").exists():
    with open(report_dir / "automl_metrics.json") as f:
        m = json.load(f)
        results["TPOT\n(AutoML)"] = {
            "accuracy": m.get("automl_accuracy"),
            "f1_weighted": m.get("automl_f1_weighted"),
            "f1_macro": m.get("automl_f1_macro"),
        }

print(f"Znalezione wyniki: {len(results)} modeli")
for name, m in results.items():
    print(f"  {name.replace(chr(10), ' ')}: F1w={m.get('f1_weighted', 'N/A')}")

In [ ]:
# Tabela porownawcza
if results:
    df_results = pd.DataFrame(results).T
    df_results = df_results[["accuracy", "f1_weighted", "f1_macro"]].astype(float)
    df_results.columns = ["Accuracy", "F1 Weighted", "F1 Macro"]
    df_results = df_results.sort_values("F1 Weighted", ascending=False)
    
    print("Porownanie modeli (posortowane wg F1 Weighted):")
    display(df_results.round(4))
    
    # Wykres porownawczy
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    for i, metric in enumerate(["Accuracy", "F1 Weighted", "F1 Macro"]):
        colors = sns.color_palette("viridis", len(df_results))
        df_results[metric].plot(kind="barh", ax=axes[i], color=colors)
        axes[i].set_title(metric, fontsize=14)
        axes[i].set_xlim(0, 1)
        for j, v in enumerate(df_results[metric]):
            axes[i].text(v + 0.01, j, f"{v:.4f}", va="center", fontsize=10)
    
    plt.suptitle("Porownanie modeli", fontsize=16, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print("Brak wynikow - uruchom pipeline'y: kedro run --pipeline=tuning")

## Confusion matrix najlepszego modelu

In [ ]:
# Zaladowanie najlepszego modelu i ewaluacja
best_model_path = models_dir / "tuned_model.pkl"
if not best_model_path.exists():
    best_model_path = models_dir / "best_comparison_model.pkl"
if not best_model_path.exists():
    best_model_path = models_dir / "model.pkl"

if best_model_path.exists():
    with open(best_model_path, "rb") as f:
        best_model = pickle.load(f)
    
    # Wczytaj przetworzone dane
    features_path = Path("../data/03_primary/crash_features.parquet")
    if features_path.exists():
        df_feat = pd.read_parquet(features_path)
        X = df_feat.drop("Severity_Group", axis=1)
        y = df_feat["Severity_Group"]
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )
        
        preds = best_model.predict(X_test)
        labels = sorted(y_test.unique())
        
        # Confusion matrix
        fig, ax = plt.subplots(figsize=(8, 6))
        cm = confusion_matrix(y_test, preds, labels=labels)
        sns.heatmap(cm, annot=True, fmt="d", cmap="YlOrRd",
            xticklabels=labels, yticklabels=labels, ax=ax)
        ax.set_title(f"Confusion Matrix - Best Model ({best_model_path.stem})", fontsize=14)
        ax.set_xlabel("Predicted")
        ax.set_ylabel("Actual")
        plt.tight_layout()
        plt.show()
        
        print(f"\nClassification Report ({best_model_path.stem}):")
        print(classification_report(y_test, preds))
        
        # Feature importance
        if hasattr(best_model, "feature_importances_"):
            fi = pd.Series(best_model.feature_importances_, index=X.columns).sort_values(ascending=False)
            fig, ax = plt.subplots(figsize=(12, 8))
            fi.head(20).plot(kind="barh", ax=ax, color=sns.color_palette("viridis", 20))
            ax.set_title("Top 20 Feature Importances (Best Model)")
            ax.invert_yaxis()
            plt.tight_layout()
            plt.show()
else:
    print("Brak wytrenowanego modelu - uruchom: kedro run")